##### `Semantic Search with Pinecone and embeddings using HuggingFace Model`

In [1]:
# Import Libraries

import pandas as pd
import os
from tqdm import tqdm
from dotenv import  load_dotenv
from pinecone import Pinecone , ServerlessSpec
from sentence_transformers import SentenceTransformer

c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dotenv file

_ = load_dotenv(override=True)
pinecone_key = os.getenv("PINECONE_API_KEY")


In [3]:
file_path = os.path.join(os.getcwd() , "data" , "articles_new.csv")
df = pd.read_csv(file_path)

# Add another clolumn (Just an exmaple) --> to be used as metadata
df['class'] = ['class-a', 'class-b'] * 250
df

,title,id,class
0,Mental Note Vol. 24,3054,class-a
1,Your Brain On Coronavirus,3055,class-b
2,Mind Your Nose,3056,class-a
3,The 4 Purposes of Dreams,3057,class-b
4,Surviving a Rod Through the Head,3058,class-a
...,...,...,...
495,Is It Worth to Invest In Mobile E-commerce App...,3549,class-b
496,Let go of these things for a happier 2021,3550,class-a
497,Not Everyone Will like Your Writing,3551,class-b
498,Is Technology Neutral?,3552,class-a


* `Embeddings using HuggingFace Model`

In [4]:
# The Model
model_hugging_6 = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2" , device="cpu")
model_hugging_12 = SentenceTransformer(model_name_or_path="all-MiniLM-L12-v2" , device="cpu")


In [5]:
# test model for embedding 

vect_length_hugging_6 = len(model_hugging_6.encode(df["title"].iloc[0]))
print('Length of Hugging Face "all-MiniLM-L6-v2" Model is:', vect_length_hugging_6)

model_hugging_6.encode(df["title"].iloc[0])[:10]


Length of Hugging Face "all-MiniLM-L6-v2" Model is: 384


array([-0.0131355 ,  0.06555429, -0.01977038, -0.03618843, -0.07723404,
        0.10825919,  0.08496749,  0.02290436,  0.02159446,  0.00035314],
      dtype=float32)

In [6]:
# test model for embedding 
vect_length_hugging_12 = len(model_hugging_12.encode(df['title'].iloc[0]))
print('Length of Hugging Face "all-MiniLM-L12-v2" Model is:', vect_length_hugging_12)

# First 10 values
model_hugging_12.encode(df['title'].iloc[0])[:10]

Length of Hugging Face "all-MiniLM-L12-v2" Model is: 384


array([-0.00662944,  0.05047883, -0.02623439,  0.02983774,  0.01254785,
        0.00395187,  0.04635642,  0.01441403,  0.02872927,  0.05269251],
      dtype=float32)

* `Pinecone in Code`

In [8]:
# Connect to Pinecone
pc = Pinecone(api_key="pcsk_4oYw5X_QJd4vNAdAumcCFGqWdidLZaMxRgUrsai5GyLXWCRDLfsMFD5bDxrYcQ2yxjjNhY")

# Create the Index
index_name = 'quickstart'
try:
    pc.create_index(
            name=index_name,
            dimension=vect_length_hugging_6, 
            metric='cosine',              
            spec=ServerlessSpec(
                cloud='aws',
                region='us-east-1'),
            deletion_protection='enabled'
                )
except:
    pass

index = pc.Index(name=index_name)

* `Using all-MiniLM-L6-v2`

In [10]:
# Looping over the Dataset and upsert through batches
batch_size = 16
failed_ids = []

for batch_start in tqdm(range(0, len(df), batch_size)):
    try:
        # Prepare Batches
        batch_end = min(batch_start+batch_size, len(df))                            ## to handle the end of each batch
        titles_batch = df['title'][batch_start: batch_end].tolist()                 ## Slice the DF according to each batch
        ids_batch = df['id'][batch_start: batch_end].astype(str).tolist()           ## Also, Slice for the Ids according to each batch
        metadata_batch = df['class'][batch_start: batch_end].tolist()

        # Get Embeddings using HuggingFace model
        embeds_batch = model_hugging_6.encode(titles_batch).tolist()

        # Prepare data for Pinecone upsert
        to_upsert = [(id, emb, {'class': cls})
                            for id, emb, cls in zip(ids_batch, embeds_batch, metadata_batch)]
        # Insert to pinecone
        _ = index.upsert(vectors=to_upsert, namespace='HF:all-MiniLM-L6-v2')
    
    except Exception as e:
        print(f'Error Upserting: {e}')
        failed_ids.append(ids_batch)

100%|██████████| 32/32 [00:38<00:00,  1.19s/it]


* `Using all-MiniLM-L12-v2`

In [11]:
# Looping over the Dataset and upsert through batches
batch_size = 16
failed_ids = []

for batch_start in tqdm(range(0, len(df), batch_size)):
    try:
        # Prepare Batches
        batch_end = min(batch_start+batch_size, len(df))                            ## to handle the end of each batch
        titles_batch = df['title'][batch_start: batch_end].tolist()                 ## Slice the DF according to each batch
        ids_batch = df['id'][batch_start: batch_end].astype(str).tolist()           ## Also, Slice for the Ids according to each batch
        metadata_batch = df['class'][batch_start: batch_end].tolist()

        # Get Embeddings using HuggingFace model
        embeds_batch = model_hugging_12.encode(titles_batch).tolist()

        # Prepare data for Pinecone upsert
        to_upsert = [(id, emb, {'class': cls})
                             for id, emb, cls in zip(ids_batch, embeds_batch, metadata_batch)]

        # Insert to pinecone
        _ = index.upsert(vectors=to_upsert, namespace='HF:all-MiniLM-L12-v2')
    
    except Exception as e:
        print(f'Error Upserting: {e}')
        failed_ids.append(ids_batch)

100%|██████████| 32/32 [00:42<00:00,  1.32s/it]


In [12]:
# Inference (Query in real-time) (you can make more than query in one, List)
query_text = 'Neutral Technology'

# Generate Embedding for the query_text
query_embedding = model_hugging_6.encode(query_text).tolist()

# Search in pinecone
results = index.query(vector=query_embedding, 
                    top_k=5, include_metadata=True, 
                    namespace='HF:all-MiniLM-L6-v2', 
                    filter={'class': 'class-b'})

# Results
results = results['matches']
results

[{'id': '3393',
  'metadata': {'class': 'class-b'},
  'score': 0.345279366,
  'values': []},
 {'id': '3107',
  'metadata': {'class': 'class-b'},
  'score': 0.330305696,
  'values': []},
 {'id': '3311',
  'metadata': {'class': 'class-b'},
  'score': 0.225047886,
  'values': []},
 {'id': '3277',
  'metadata': {'class': 'class-b'},
  'score': 0.218963593,
  'values': []},
 {'id': '3441',
  'metadata': {'class': 'class-b'},
  'score': 0.205456868,
  'values': []}]

In [13]:
df

,title,id,class
0,Mental Note Vol. 24,3054,class-a
1,Your Brain On Coronavirus,3055,class-b
2,Mind Your Nose,3056,class-a
3,The 4 Purposes of Dreams,3057,class-b
4,Surviving a Rod Through the Head,3058,class-a
...,...,...,...
495,Is It Worth to Invest In Mobile E-commerce App...,3549,class-b
496,Let go of these things for a happier 2021,3550,class-a
497,Not Everyone Will like Your Writing,3551,class-b
498,Is Technology Neutral?,3552,class-a


In [14]:
# You can delete vectors using ids
_ = index.delete(ids=['3054', '3055'], namespace='HF:all-MiniLM-L6-v2')

In [15]:
# To update the embeddings of any id 
text_update = 'This is for updating the Id and change embeddings'
embeds_update = model_hugging_6.encode(text_update).tolist()

# Update or you can use upsert
_ = index.update(id='3191', values=embeds_update, namespace='HF:all-MiniLM-L6-v2')

In [16]:
## Fetch ids
index.fetch(ids=['3191', '3292'], namespace='HF:all-MiniLM-L6-v2')

FetchResponse(namespace='HF:all-MiniLM-L6-v2', vectors={'3292': Vector(id='3292', values=[-0.0587002821, -0.10953746, 0.0402981602, 0.0467616059, -0.0353818387, 0.0857539177, 0.0412121937, 0.0141806751, 0.00177014631, -0.0770865381, -0.0415372662, -0.0538043194, 0.0217062514, -0.0458817147, 0.0705075711, 0.0759797543, 0.0345984846, -0.100260712, 0.0140041336, -0.0515077859, 0.0896312445, -0.0768813, -0.0563248619, 0.0224374942, 0.0307733864, 0.0182192344, -0.0775668919, 0.0135552846, 0.0408200435, 0.0695489794, -0.00704766717, 0.0247531645, 0.098920472, 0.0252510887, -0.0482797, -0.00308723119, -0.00176328793, 0.0375144891, -0.0727425739, -0.00923572388, -0.00266611623, -0.0201740898, 0.0342055634, 0.0185199194, -0.0696112886, -0.0182932913, 0.00653282925, -0.0319156423, -0.043846719, 0.0264299214, 0.0211658198, -0.0810257196, -0.00519322, -0.0359051116, 0.00282149389, 0.0696480945, 0.00912587531, 0.0965562686, 0.0323681124, 0.0251859184, 0.027824318, -0.0450573228, 0.00154787372, -0.0